In [ ]:
session.sql("""
DROP TABLE IF EXISTS tmp_persons_stream_snapshot
""").collect()

In [ ]:
session.sql("""
CREATE TEMP TABLE tmp_persons_stream_snapshot AS
SELECT *
FROM STREAM_T_PERSONS
WHERE METADATA$ACTION='INSERT'
""").collect()

persons_raw = session.table("tmp_persons_stream_snapshot")
persons_raw = persons_raw.drop("METADATA$ACTION", "METADATA$ISUPDATE", "METADATA$ROW_ID")
print(f"snapshot persons records found")

In [ ]:
persons_clean = persons_raw

# Name Columns
persons_clean = persons_clean.with_column(
    "LAST_NAME",
    when(trim(col("LAST_NAME")) == "", lit("NA"))
    .otherwise(coalesce(trim(col("LAST_NAME")), lit("NA")))
)

persons_clean = persons_clean.with_column(
    "FIRST_NAME",
    when(trim(col("FIRST_NAME")) == "", lit("NA"))
    .otherwise(coalesce(trim(col("FIRST_NAME")), lit("NA")))
)

persons_clean = persons_clean.with_column(
    "MIDDLE_NAME",
    when(trim(col("MIDDLE_NAME")) == "", lit("NA"))
    .otherwise(coalesce(trim(col("MIDDLE_NAME")), lit("NA")))
)

# Indicator Columns
indicator_cols = [
    "DO_NOT_RESUSCITATE_IND",
    "EDUCATIONAL_PARENT_IND",
    "GUARDIANSHIP_CONSENT_IND",
    "ADOPTION_NOT_APPROPRIATE_IND",
    "BIO_PARENTS_RETURN_IND",
    "BIRTH_CERTIFICATE_AVLBL_IND",
    "NO_PHONE_IND",
    "PET_OWNER_IND",
    "SMOKER_IND",
    "SSN_VALIDATION_IND",
    "RESTRICTED_IND",
    "ADOPTION_MATCH_FOUND_IND",
    "APPROXIMATE_DOB_IND",
    "APPROX_LAST_ADOPTED_DATE_IND"
]

for c in indicator_cols:
    persons_clean = persons_clean.with_column(
        c,
        when(trim(col(c)) == "", lit("N"))
        .otherwise(coalesce(trim(col(c)), lit("N")))
    )

# User Columns
persons_clean = persons_clean.with_column(
    "ADD_USER",
    when(trim(col("ADD_USER")) == "", lit("SYSTEM"))
    .otherwise(coalesce(trim(col("ADD_USER")), lit("SYSTEM")))
)

persons_clean = persons_clean.with_column(
    "MOD_USER",
    when(trim(col("MOD_USER")) == "", lit("SYSTEM"))
    .otherwise(coalesce(trim(col("MOD_USER")), lit("SYSTEM")))
)

# Trim Name Columns
for c in [
    "UC_LAST_NAME",
    "UC_FIRST_NAME",
    "UC_MIDDLE_NAME",
    "SDX_LAST_NAME",
    "SDX_FIRST_NAME",
    "SDX_MIDDLE_NAME"
]:
    persons_clean = persons_clean.with_column(c, trim(col(c)))

# Rename RAW load timestamp
persons_clean = persons_clean.with_column_renamed(
    "LOAD_TIMESTAMP",
    "RAW_LOAD_TIMESTAMP"
)

print("persons clean")

In [ ]:
valid_persons = persons_clean.filter(
    col("PERSON_ID").is_not_null() &
    col("FIRST_NAME").is_not_null() &
    col("LAST_NAME").is_not_null()
)

invalid_persons = persons_clean.filter(
    col("PERSON_ID").is_null() |
    col("FIRST_NAME").is_null() |
    col("LAST_NAME").is_null()
).with_column(
    "ERROR_REASON",
    when(col("PERSON_ID").is_null(), lit("PERSON_ID cannot be NULL"))
    .when(col("FIRST_NAME").is_null(), lit("FIRST_NAME cannot be NULL"))
    .when(col("LAST_NAME").is_null(), lit("LAST_NAME cannot be NULL"))
)
print("seperated valid and invalid records")

In [ ]:
persons_final = valid_persons.with_column(
    "CHECKSUM",
    sha2(
        concat_ws(
            lit("|"),

            # Primary Key
            coalesce(col("PERSON_ID").cast("string"), lit("")),

            # Names
            coalesce(col("LAST_NAME"), lit("")),
            coalesce(col("FIRST_NAME"), lit("")),
            coalesce(col("MIDDLE_NAME"), lit("")),
            coalesce(col("UC_LAST_NAME"), lit("")),
            coalesce(col("UC_FIRST_NAME"), lit("")),
            coalesce(col("UC_MIDDLE_NAME"), lit("")),
            coalesce(col("SDX_LAST_NAME"), lit("")),
            coalesce(col("SDX_FIRST_NAME"), lit("")),
            coalesce(col("SDX_MIDDLE_NAME"), lit("")),

            # Codes
            coalesce(col("CITIZENSHIP_CODE"), lit("")),
            coalesce(col("PRIMARY_LANGUAGE_CODE"), lit("")),
            coalesce(col("MARITAL_STATUS_CODE"), lit("")),
            coalesce(col("PREFIX_CODE"), lit("")),
            coalesce(col("SUFFIX_CODE"), lit("")),
            coalesce(col("PROFESSIONAL_SUFFIX_CODE"), lit("")),
            coalesce(col("AGE_UNIT_CODE"), lit("")),
            coalesce(col("BIRTH_PLACE_STATE_CODE"), lit("")),
            coalesce(col("BIRTH_PLACE_COUNTRY_CODE"), lit("")),
            coalesce(col("DECEASED_COUNTRY_CODE"), lit("")),
            coalesce(col("DECEASED_STATE_CODE"), lit("")),
            coalesce(col("CURRENT_GRADE_CODE"), lit("")),
            coalesce(col("ETHNICITY_CODE"), lit("")),
            coalesce(col("GENDER_CODE"), lit("")),
            coalesce(col("IMMIGRATION_STATUS_CODE"), lit("")),
            coalesce(col("RACE_CODE"), lit("")),
            coalesce(col("RELIGION_CODE"), lit("")),
            coalesce(col("SSN_SOURCE_CODE"), lit("")),
            coalesce(col("CHILD_ADOPTED_CODE"), lit("")),
            coalesce(col("AGE_CHILD_ADP_UNIT_CODE"), lit("")),
            coalesce(col("DUP_PERSON_STATUS_CODE"), lit("")),

            # Indicator Columns
            coalesce(col("DO_NOT_RESUSCITATE_IND"), lit("")),
            coalesce(col("EDUCATIONAL_PARENT_IND"), lit("")),
            coalesce(col("GUARDIANSHIP_CONSENT_IND"), lit("")),
            coalesce(col("ADOPTION_NOT_APPROPRIATE_IND"), lit("")),
            coalesce(col("BIO_PARENTS_RETURN_IND"), lit("")),
            coalesce(col("BIRTH_CERTIFICATE_AVLBL_IND"), lit("")),
            coalesce(col("NO_PHONE_IND"), lit("")),
            coalesce(col("PET_OWNER_IND"), lit("")),
            coalesce(col("SMOKER_IND"), lit("")),
            coalesce(col("SSN_VALIDATION_IND"), lit("")),
            coalesce(col("RESTRICTED_IND"), lit("")),
            coalesce(col("ADOPTION_MATCH_FOUND_IND"), lit("")),
            coalesce(col("APPROXIMATE_DOB_IND"), lit("")),
            coalesce(col("APPROX_LAST_ADOPTED_DATE_IND"), lit("")),

            # User columns
            coalesce(col("ADD_USER"), lit("")),
            coalesce(col("MOD_USER"), lit("")),

            # Business attributes
            coalesce(col("AGE_NUM").cast("string"), lit("")),
            coalesce(col("BIRTH_DATE").cast("string"), lit("")),
            coalesce(col("BIRTH_PLACE_CITY_NAME"), lit("")),
            coalesce(col("BIRTH_PLACE_HOSPITAL_NAME"), lit("")),
            coalesce(col("DECEASED_DATE").cast("string"), lit("")),
            coalesce(col("DECEASED_CITY_NAME"), lit("")),
            coalesce(col("ALIEN_REGISTRATION_NUM"), lit("")),
            coalesce(col("OCCUPATION_DESC"), lit("")),
            coalesce(col("SOCIAL_SECURITY_NUM"), lit("")),
            coalesce(col("ITIN_NUM"), lit("")),
            coalesce(col("BIRTH_DELIVERY_CMNT"), lit("")),
            coalesce(col("BIRTH_ORDER_SEQUENCE_NUM").cast("string"), lit("")),
            coalesce(col("MARITAL_STATUS_EFFECT_DATE").cast("string"), lit("")),
            coalesce(col("COMMON_CLIENT_ID").cast("string"), lit("")),
            coalesce(col("AGE_CHILD_ADOPTED_NUM").cast("string"), lit("")),
            coalesce(col("GESTATIONAL_AGE_NUM").cast("string"), lit("")),
            coalesce(col("US_ENTRY_DATE").cast("string"), lit("")),
            coalesce(col("ADOPTION_FREED_DATE").cast("string"), lit("")),
            coalesce(col("ARCHIVE_DATE").cast("string"), lit("")),
            coalesce(col("ARCHIVE_FILE_NAME"), lit("")),
            coalesce(col("LAST_ELECTRON_RETRIEVAL_DATE").cast("string"), lit("")),
            coalesce(col("PERSON_PERSON_REQUESTS_ID").cast("string"), lit("")),
            coalesce(col("AFCARS_ADOPTION_SEND_DATE").cast("string"), lit("")),
            coalesce(col("PERSON_PERSON_PRIMARY_ID").cast("string"), lit("")),
            coalesce(col("LICENSE_RENEWAL_TICKLER_ID").cast("string"), lit("")),
            coalesce(col("BI_ANNUAL_REASSESS_TICKLER_ID").cast("string"), lit("")),
            coalesce(col("CHILD_ADOPTED_DATE").cast("string"), lit("")),
            coalesce(col("HEALTH_CERT_TICKLER_ID").cast("string"), lit("")),
            coalesce(col("REEVALUATION_TICKLER_ID").cast("string"), lit("")),
            coalesce(col("BIMONTHLY_VISIT_TICKLER_ID").cast("string"), lit("")),
            coalesce(col("TICKLER_ID").cast("string"), lit("")),
            coalesce(col("SSN_REFUSAL_DATE").cast("string"), lit(""))
        ),
        256
    )
)
print("checksum calculation completed")

In [ ]:
persons_final = persons_final.with_column(
    "STAGING_LOADED_TIMESTAMP",
    current_timestamp()
)

persons_final.write.save_as_table(
    table_name=f"{DB}.{STG}.{STG_PERSONS}",
    mode="truncate"
)

print(f"PERSONS loaded into {STG}.{STG_PERSONS}")

In [ ]:
invalid_persons.create_or_replace_temp_view("tmp_invalid_persons")

invalid_count = invalid_persons.count()

if invalid_count > 0:

    session.sql(f"""
    INSERT INTO {DB}.{STG}.{INVALID_RECORDS}
    (
        TABLE_NAME,
        ERROR_REASON,
        SOURCE_FILE_NAME,
        RAW_LOAD_TIMESTAMP,
        RAW_RECORD
    )
    SELECT
        'T_PERSONS',
        ERROR_REASON,
        SOURCE_FILE_NAME,
        RAW_LOAD_TIMESTAMP,
        OBJECT_CONSTRUCT(*)
    FROM tmp_invalid_persons
    """).collect()

    print(f"Invalid PERSONS records stored")

else:
    print("No invalid PERSONS records")

In [ ]:
rows_processed = persons_final.count()
rows_failed = invalid_persons.count()

session.call(
    f"{DB}.{AUDIT}.{SP_LOG_AUDIT}",
    session.sql("SELECT UUID_STRING()").collect()[0][0],
    "STG_PERSONS_LOAD",
    "STAGING",
    datetime.now(),
    datetime.now(),
    rows_processed,
    rows_failed,
    "SUCCESS",
    None
)
session.call(
    f"{DB}.{UTIL}.{SP_SEND_PIPELINE_NOTIFICATION}",
    "SUCCESS",
    "STG_PERSONS_LOAD",
    "STAGING",
    f"PERSONS staging completed. Rows processed: {rows_processed}, Failed rows: {rows_failed}"
)
print("Audit log inserted and email sent")